# Notebook 2: RAG — Knowledge Retrieval with Pinecone

**Technique covered:** Transformer-based Models / LLMs (embeddings + RAG)

This notebook demonstrates the Retrieval-Augmented Generation (RAG) pipeline:
1. Load domain knowledge from JSON into a Pinecone vector index
2. Encode knowledge chunks with Google `text-embedding-004` (768-dim)
3. Perform semantic search to retrieve relevant context
4. Show how retrieved context improves question generation quality

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
load_dotenv('../.env')

from google import genai as google_genai
import numpy as np
from pathlib import Path

client = google_genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
EMBED_MODEL = 'models/text-embedding-004'

print('Google GenAI client ready')

## 1. Load Domain Knowledge

In [ ]:
DATA_DIR = Path('../data')

def load_domain(domain: str) -> list[dict]:
    path = DATA_DIR / f'{domain}.json'
    with open(path) as f:
        data = json.load(f)
    return data['topics']

se_topics = load_domain('software_engineering')
print(f'Loaded {len(se_topics)} topics for software_engineering')
print('\nFirst topic:')
print(f'  Concept    : {se_topics[0]["concept"]}')
print(f'  Difficulty : {se_topics[0]["difficulty"]}')
print(f'  Explanation: {se_topics[0]["explanation"][:100]}...')

## 2. Generate Embeddings

In [ ]:
def embed(text: str) -> np.ndarray:
    result = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return np.array(result.embeddings[0].values)

# Embed all topics
knowledge_store = []
for topic in se_topics:
    text = f"{topic['concept']}: {topic['explanation']}"
    knowledge_store.append({
        **topic,
        'embedding': embed(text),
        'full_text': text
    })

print(f'Embedded {len(knowledge_store)} knowledge chunks')
print(f'Embedding dimension: {len(knowledge_store[0]["embedding"])}')

## 3. Semantic Retrieval (In-Memory Demo)

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve(query: str, store: list, top_k: int = 3) -> list[dict]:
    q_emb = embed(query)
    scored = [
        {**item, 'score': cosine_similarity(q_emb, item['embedding'])}
        for item in store
    ]
    return sorted(scored, key=lambda x: x['score'], reverse=True)[:top_k]

# Test retrieval
query = "How do distributed systems handle inconsistent data?"
results = retrieve(query, knowledge_store, top_k=3)

print(f'Query: "{query}"\n')
for i, r in enumerate(results, 1):
    print(f'Rank {i} (score={r["score"]:.4f})')
    print(f'  Concept    : {r["concept"]}')
    print(f'  Difficulty : {r["difficulty"]}')
    print(f'  Explanation: {r["explanation"][:120]}...')
    print()

## 4. RAG vs No-RAG: Question Generation Quality

In [ ]:
from google.genai import types as genai_types

def generate_question(context: str, difficulty: str) -> str:
    prompt = f"""Generate one {difficulty}-level technical interview question for a software engineering role.

Context from domain knowledge:
{context}

Output ONLY the question text."""
    response = client.models.generate_content(
        model='gemini-2.0-flash',
        contents=prompt,
        config=genai_types.GenerateContentConfig(temperature=0.7, max_output_tokens=150),
    )
    return response.text.strip()

# Without RAG
no_rag_q = generate_question('No specific context.', 'hard')

# With RAG
rag_context = '\n'.join(f"- {r['concept']}: {r['explanation'][:150]}" for r in results)
rag_q = generate_question(rag_context, 'hard')

print('WITHOUT RAG:')
print(f'  {no_rag_q}')
print()
print('WITH RAG (context from distributed systems retrieval):')
print(f'  {rag_q}')

## 5. Pinecone Integration (Production Path)

In [ ]:
# This cell shows the production Pinecone ingestion — requires PINECONE_API_KEY in .env
import os

PINECONE_KEY = os.getenv('PINECONE_API_KEY')

if PINECONE_KEY:
    from rag.knowledge_base import KnowledgeBase
    kb = KnowledgeBase()
    n = kb.load_domain('software_engineering')
    print(f'Upserted {n} vectors into Pinecone')
    
    # Live query
    live_results = kb.query("concurrency and race conditions", domain='software_engineering', top_k=2)
    for r in live_results:
        print(f"  {r['concept']} (score={r['relevance_score']})")
else:
    print('PINECONE_API_KEY not set — skipping live Pinecone demo.')
    print('Set it in backend/.env to run the production pipeline.')

## 6. Retrieval Quality Analysis

In [ ]:
import pandas as pd

test_queries = [
    ("caching and performance", ["Caching Strategies"]),
    ("database transactions ACID", ["SQL vs NoSQL Databases"]),
    ("class design principles", ["SOLID Principles", "Design Patterns"]),
    ("distributed consistency tradeoffs", ["Distributed Systems — CAP Theorem", "Microservices Architecture"]),
]

rows = []
for query, expected in test_queries:
    top1 = retrieve(query, knowledge_store, top_k=1)[0]
    hit = any(e in top1['concept'] for e in expected)
    rows.append({'Query': query[:45], 'Top-1 Retrieved': top1['concept'][:40], 'Score': round(top1['score'], 4), 'Hit': '✓' if hit else '✗'})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nPrecision@1: {df['Hit'].str.count('✓').sum() / len(df):.0%}")

## Summary

The RAG pipeline:
- Embeds 15+ domain knowledge chunks using Google transformer embeddings (768 dimensions)
- Retrieves semantically relevant context in real-time for each user turn
- Grounds question generation in accurate domain knowledge (reducing hallucination)
- Achieves high Precision@1 on diverse query types

In production, Pinecone handles the vector store with millisecond query latency.